# Week 11, Day 3: The discrete Fourier transform, the FFT and sampling

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w011/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The DFT as a matrix product

Write `dft_matrix(N)`, the matrix whose (k, n) entry is $e^{-i2\pi kn/N}$, so that the DFT is a matrix-vector product. Check it against `np.fft.fft` and check Parseval's theorem for a random signal (`lhs` and `rhs`). Then time both methods for N = 1024.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

def dft_matrix(N):
    n = np.arange(N)
    return ...

x = np.random.default_rng(1).normal(size=256)
X = dft_matrix(256) @ x
lhs, rhs = np.sum(np.abs(x) ** 2), np.sum(np.abs(X) ** 2) / 256

big = np.random.default_rng(2).normal(size=1024)
t0 = time.perf_counter(); dft_matrix(1024) @ big; t_mat = time.perf_counter() - t0
t0 = time.perf_counter()
for _ in range(100):
    np.fft.fft(big)
t_fft = (time.perf_counter() - t0) / 100
print("Parseval:", round(lhs, 4), round(rhs, 4), f"   matrix {1000 * t_mat:.2f} ms, FFT {1000 * t_fft:.3f} ms")

In [ ]:
import numpy as np
def test_matrix():
    "dft_matrix has entries exp(-2 pi i n k / N)"
    F = dft_matrix(4)
    assert np.allclose(F, np.exp(-2j * np.pi * np.outer(np.arange(4), np.arange(4)) / 4))

def test_match():
    "the matrix DFT matches np.fft.fft"
    x = np.random.default_rng(0).normal(size=64)
    assert np.allclose(dft_matrix(64) @ x, np.fft.fft(x))

def test_parseval():
    "Parseval holds: sum |x|^2 equals sum |X|^2 / N"
    assert np.isclose(lhs, rhs)

test_matrix()
test_match()
test_parseval()
print("All checks passed")

### Exercise: Find two rhythms in hertz

A 2-second field potential sampled at 250 Hz contains a 6 Hz theta rhythm of amplitude 1 and a 40 Hz gamma rhythm of amplitude 0.4, plus noise. Compute `freqs` with `rfftfreq` and `amps`, the amplitude spectrum, and plot it. Store the frequency resolution in `df`.

In [ ]:
rng = np.random.default_rng(1965)
fs = 250
t = np.arange(0, 2, 1 / fs)
x = 1.0 * np.sin(2 * np.pi * 6 * t) + 0.4 * np.sin(2 * np.pi * 40 * t + 1.0) + 0.3 * rng.normal(size=len(t))

freqs = ...
amps = ...
df = ...

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(t, x, lw=0.8); ax[0].set_xlim(0, 0.5); ax[0].set_xlabel("time (s)")
ax[1].plot(freqs, amps); ax[1].set_xlabel("frequency (Hz)"); ax[1].set_ylabel("amplitude"); ax[1].set_xlim(0, 80)
plt.tight_layout(); plt.show()
print("frequency resolution", df, "Hz;  Nyquist", fs / 2, "Hz")

In [ ]:
import numpy as np
def test_axes():
    "freqs and amps come from rfft and rfftfreq, with amplitude scaling 2/N"
    assert np.allclose(freqs, np.fft.rfftfreq(len(x), 1 / fs))
    assert np.allclose(amps, 2 * np.abs(np.fft.rfft(x)) / len(x))

def test_peaks():
    "the two largest peaks are at 6 and 40 Hz with amplitudes near 1.0 and 0.4"
    top = np.argsort(amps)[-2:]
    assert sorted(np.round(freqs[top])) == [6.0, 40.0]
    assert abs(amps[np.argmin(np.abs(freqs - 6))] - 1.0) < 0.1 and abs(amps[np.argmin(np.abs(freqs - 40))] - 0.4) < 0.1

def test_resolution():
    "df is the frequency resolution for this recording"
    assert np.isclose(df, 0.5)

test_axes()
test_peaks()
test_resolution()
print("All checks passed")

### Exercise: Catch an alias

Sample a 90 Hz sinusoid at 100 Hz for 2 s and find the frequency of the largest peak in its spectrum (`apparent`). Then write `alias_freq(f, fs)` that predicts where any frequency will appear.

In [ ]:
fs_low = 100
t_low = np.arange(0, 2, 1 / fs_low)
y = np.sin(2 * np.pi * 90 * t_low)
f_low = np.fft.rfftfreq(len(y), 1 / fs_low)
apparent = ...

def alias_freq(f, fs):
    return ...

print("90 Hz sampled at 100 Hz appears at", apparent, "Hz;  predictions:", [alias_freq(f, 100) for f in [40, 60, 90, 130]])

In [ ]:
import numpy as np
def test_peak():
    "the 90 Hz oscillation sampled at 100 Hz appears at 10 Hz"
    assert np.isclose(apparent, 10.0)

def test_formula():
    "alias_freq gives |f - m fs| for the nearest whole m"
    assert np.isclose(alias_freq(90, 100), 10) and np.isclose(alias_freq(130, 100), 30) and np.isclose(alias_freq(40, 100), 40)

test_peak()
test_formula()
print("All checks passed")